## Pipeline for 65-HD-sEMG dataset

### Model trained using 65HDsEMG data

+ 128 window-size frames
+ 64 channels
+ normalization per channel using MinMax



In [1]:
import os, sys

processing_source_path = os.path.abspath('./../../../Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
import ProcessingPipeline as pp

dataloader_source_path = os.path.abspath('./../../../Dataloader/')
if(dataloader_source_path not in sys.path):
    sys.path.append(dataloader_source_path)
from EMGGestureDataGen import EMGGestureDataGen

dataloader_source_path = os.path.abspath('./../../../Dataloader/')
if(dataloader_source_path not in sys.path):
    sys.path.append(dataloader_source_path)
from EMGGestureDataGen import EMGGestureDataGen
from KerasDataPipeline import KerasDataPipeline

model_source_path = os.path.abspath('./../../../Keras_deep_models/')
if(model_source_path not in sys.path):
    sys.path.append(model_source_path)
from EMGConv2DResNetModel import EMGConv2DResNetModel

dataloader_source_path = os.path.abspath('./../../../Utils/')
if(dataloader_source_path not in sys.path):
    sys.path.append(dataloader_source_path)
import utils as utils
import available_metrics as metrics

In [2]:
import tensorflow as tf
import sklearn.metrics as skm
import numpy as np
import glob
import scipy.io
import time
import gc
import h5py
import pandas as pd

In [3]:
import utils as utils

In [4]:
import matplotlib.pyplot as plt

In [5]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

2025-08-10 16:26:23.219173: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-08-10 16:26:23.219212: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:163] no NVIDIA GPU device is present: /dev/nvidia0 does not exist


In [6]:
params = {
    'dataset_path': '/root/data/data/65_HDsEMG/',
    'window': 128,
    'step': 32,
    'preprocessing': '4step',
    'num_channels': 64,
    'num_classes':  9,
    'num_samples': 256,
    'num_windows': 10,
    'img_height': 8,
    'img_width': 16,
    'print_model': True,
    'num_epochs':60,
    # 'gesture_list' : [2, 20, 21, 26, 27, 30 ,31, 33, 34],
    'gesture_list': [0, 13, 14, 17, 20, 36, 58, 59, 63, 64], # 65-HD-sEMG list
    'n_voting': 3,
    'armband': False,
    'MAFilter': 32,
    'plot_data': False
    
}

## Buffering data

In [7]:
def buffer_data(data, new_batch):
    data = np.roll(data,-new_batch.shape[1],axis=1)
    data[:,-new_batch.shape[1]:] = new_batch
    
    return data

## Preprocessing

In [8]:
def EMGTKE(data):
    return data[:,1:-1]*data[:,1:-1] - data[:,:-2]*data[:,2:]


In [9]:
def retification(data):
    return np.abs(data)

In [10]:
def MAFilter(data,L):
        
    cs = np.cumsum(data,axis=1,dtype=float)
    cs[:,L:] = cs[:,L:] - cs[:,:-L]
    
    return cs[:,L-1:]/L 


In [11]:

def norm(data,ax):

    samples_min = data.min(axis=ax,keepdims=True)
    data = (data - samples_min)
    samples_max = data.max(axis=ax,keepdims=True) + sys.float_info.epsilon
    data = data / samples_max


    return  data

## Noise generator

In [12]:
def band_limited_noise(min_freq, max_freq, samples=1024, samplerate=1):
    freqs = np.abs(np.fft.fftfreq(samples, 1/samplerate))
    f = np.zeros(samples)
    idx = np.where(np.logical_and(freqs>=min_freq, freqs<=max_freq))[0]
    f[idx] = 1
    return fftnoise(f)

In [13]:
def fftnoise(f):
    f = np.array(f, dtype='complex')
    Np = (len(f) - 1) // 2
    phases = np.random.rand(Np) * 2 * np.pi
    phases = np.cos(phases) + 1j * np.sin(phases)
    f[1:Np+1] *= phases
    f[-1:-1-Np:-1] = np.conj(f[1:Np+1])
    return np.fft.ifft(f).real

In [14]:
def create_noise(minimum, maximum, data_len):
    x = band_limited_noise(10, 900,  data_len, 2048)
    x -= np.min(x)
    x /= (np.max(x))
    x *= (maximum-minimum)
    x += minimum
    return x

In [ ]:
def inject_electrode_failure(diff_data, failure_type, index1, index2, n_channels):
    """
    Simula falha de eletrodos em sinais diferenciais.

    Parameters
    ----------
    diff_data : ndarray
        Shape: (n_channels, n_samples)

    dropout_pct : float
        Percentual de eletrodos a serem desligados.
        Ex.: 0.1 = 10%

    Returns
    -------
    diff_new : ndarray
        Sinais diferenciais após a falha dos eletrodos.
    """

    # Reconstrução relativa dos sinais monopolares
    mono = np.cumsum(diff_data, axis=0)

    if failure_type == "Dead_electrodes":
        # Eletrodos mortos
        mono[index1[nn]+index2[nn]*8, :] = 0
    elif failure_type == "noise":
        for nn in np.arange(n_channels):
            mono[index1[nn]+index2[nn]*8,:] = mono[index1[nn]+index2[nn]*8,:] + create_noise(np.min(mono[index1[nn]+index2[nn]*8,:]),
                                                                np.max(mono[index1[nn]+index2[nn]*8,:])*max_var,
                                                                mono.shape[1])


    # Volta para representação diferencial
    diff_new = np.diff(
        np.vstack([
            np.zeros((1, mono.shape[1])),
            mono
        ]),
        axis=0
    )

    return diff_new

## Predict patterns

In [15]:

def predict_gesture(model_name, temp_data, y, params, index1, index2, max_var, n_channels, dataset_name):

    y_true = []
    y_pred = []
    y_cnn = []
    t_init = time.time()
    all_latent_times = []
    t_lat_mean = t_init
    latent_times={'first_bacth': 0 , 'second_bacth': 0 , 'mean': 0}
    
    model = tf.keras.models.load_model(model_name, compile = False)
    data = np.zeros((params['img_height']*params['img_width'],params['num_samples']))
    data_prev = data[:,-params['MAFilter']:].copy()
    y_cnn = np.zeros((params['n_voting']))

    if dataset_name =="Hyser":
        for nn in np.arange(n_channels):
            if max_var  == 0:
                temp_data[index1[nn]+index2[nn]*8,:] = np.zeros((temp_data.shape[1]))
            else:
                temp_data[index1[nn]+index2[nn]*8,:] = temp_data[index1[nn]+index2[nn]*8,:] + create_noise(np.min(temp_data[index1[nn]+index2[nn]*8,:]),
                                                                    np.max(temp_data[index1[nn]+index2[nn]*8,:])*max_var,
                                                                    temp_data.shape[1])
    elif dataset_name == "65_HDsEMG":
        if max_var == 0:
            temp_data = inject_electrode_failure(temp_data, "Dead_electrodes", index1, index2, n_channels)
        else:
            temp_data = inject_electrode_failure(temp_data, "noise", index1, index2, n_channels)

    for i in range(0,params['num_windows']*params['step'], params['step']):
        new_data = retification(temp_data[:,i:i+params['step']])
        data_MA = np.concatenate((data_prev, new_data),axis=1)
        data_prev = data_MA[:,-params['MAFilter']:].copy()

        data_MA = EMGTKE(data_MA)
        data_MA = MAFilter(data_MA,params['MAFilter'])

        data = buffer_data(data, data_MA)

        # data_prep = normZscaler(data,(-1))
        data_prep = norm(data,(0))
        # data_prep = norm(np.expand_dims(data,0),(0,-1))
        
        # print(np.min(data_prep), np.max(data_prep))
        data_prep = data_prep.reshape(1,params['img_width'],params['img_height'],params['num_samples'])

        if params['plot_data'] == True:
            plt.plot(data_prep[0,0,0,:])
            plt.title('after norm')
            plt.show()
        
        y_cnn = np.roll(y_cnn,-1)
        y_cnn[-1] = np.argmax(model.predict(data_prep))

        if len(y_pred) < params['n_voting']:
            y_pred += [y_cnn[-1]]
        else:
            values,count = np.unique(y_cnn,return_counts=True)
            if len(count) == params['n_voting']:
                y_pred += [y_pred[-1]]
            else:
                y_pred += [values[np.argmax(count)]]
                            
        values,count = np.unique(y[i:i+params['window']], return_counts=True)
        y_true += [values[np.argmax(count)]]

        if i == 0 :
            latent_times['first_bacth'] = time.time()-t_init
            print('latencia primeiro batch = ', latent_times['first_bacth'] )
            t_lat = time.time()
        elif i == params['step'] :
            latent_times['second_bacth'] = time.time()-t_lat
            print('latencia segundo batch = ', latent_times['second_bacth'])
        
        # get all latente time per loop interaction
        all_latent_times+=[time.time()-t_lat_mean]
        t_lat_mean = time.time()
        
        
    latent_times['mean'] = np.mean(all_latent_times)
    
    del(temp_data, y)
    gc.collect()
    
    return y_pred, y_true, latent_times

## Read all files

In [16]:
files_list = glob.glob(params['dataset_path']+'/2*')
files_list.sort()
files_list


['/root/data/data/65_HDsEMG/24350114',
 '/root/data/data/65_HDsEMG/25295225',
 '/root/data/data/65_HDsEMG/25295423',
 '/root/data/data/65_HDsEMG/25296644',
 '/root/data/data/65_HDsEMG/25296788',
 '/root/data/data/65_HDsEMG/25309355',
 '/root/data/data/65_HDsEMG/25312253',
 '/root/data/data/65_HDsEMG/25312382',
 '/root/data/data/65_HDsEMG/25313078',
 '/root/data/data/65_HDsEMG/25313324',
 '/root/data/data/65_HDsEMG/25313597',
 '/root/data/data/65_HDsEMG/25313726',
 '/root/data/data/65_HDsEMG/25323065',
 '/root/data/data/65_HDsEMG/25323104',
 '/root/data/data/65_HDsEMG/25323134',
 '/root/data/data/65_HDsEMG/25323215',
 '/root/data/data/65_HDsEMG/25323257']

## Model EMG 

### Running on all data

### Not all of them were used on training

Subjects used on training: 
+ 25295225 (2), 
+ 25295423 (3),
+ 25296644 (4),
+ 25313078 (9),
+ 25313597 (11),
+ 2532065 (13),
+ 25323134 (15),
+ 25323215 (16)

In [17]:
cont = 1

sub = 0
acc_total = 0

params['plot_data'] = False
results_list=[]
for file in files_list[1:2]:
    print(f'{cont} out of {len(files_list)} files', end='\r')
    print()

    with h5py.File(file, 'r') as hf:
        emg_selected_extensor_data = hf['emg_extensors'][...].reshape(64, -1).astype(np.float32)
        emg_selected_flexor_data = hf['emg_flexors'][...].reshape(64, -1).astype(np.float32)
        label = hf['adjusted_class'][...]

    if params['armband'] == True:
        
        idx = [np.arange(0,121,8), np.arange(1,58,8),np.arange(2,59,8), np.arange(5,62,8), np.arange(6,63,8), np.arange(7,129,8)]
        idx = np.hstack(idx)
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data), axis=0)
        emg_data = np.delete(emg_data,idx,axis=0)
        
    else:
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data),axis=0)

        
    
    idx_gestures = []
    
    for gest in params['gesture_list'][4:5]:
        idx_gestures += [np.argwhere(label[0,:] == gest)]
 
    emg_data = emg_data[:,np.vstack(idx_gestures)[:,0]]
    label = label[:,np.vstack(idx_gestures)[:,0]][0]
    
    all_gestures = list(params['gesture_list'])
    all_gestures = np.sort(all_gestures)


    GestureRemap = {g: new_g for new_g, g in enumerate(all_gestures)}
    yData = np.array([GestureRemap[g] for g in label])

    params['num_windows'] = (emg_data.shape[1]-params['step'])//params['step']
    print('num windows',params['num_windows'])#,emg_data.shape)

    print(np.min(emg_data), np.max(emg_data))
    t_init = time.time()
  
   
    model_name = "trainings/EMG_65HDsEMG_2D_CNN_128f_128ch_SCE_normminmax_bysample_adamW"
    
    index1 = np.append(np.arange(8),np.arange(8))
    index1 = np.append(index1,index1)
    index1 = np.append(index1,index1)

    index2 = np.repeat(np.arange(8),4)
    index2 = np.append(index2,np.roll(index2,-4))
    
    for max_var in np.arange(8,13):
    
        print('Máximo valor da amplitude do ruído: ', max_var/10)

        for nch in np.arange(8):

            print('Número de canais com ruído: ',nch)
            
            (y_pred, y_true, latent_times) = predict_gesture(model_name, emg_data.copy(), yData, params, index1[:nch], index2[:nch], max_var/10, nch, "65_HDsEMG" )

            t_total = time.time() - t_init
            print('tempo_total', t_total)

            # cont += 1
            acc = skm.accuracy_score(y_true, y_pred)
            print("User:",file,"Accuracy:", acc)
            
            
            # Salvar resultados
            results_list.append({
                "file": file,
                "gesture": gest,
                "max_noise_amplitude": max_var/10,
                "num_noisy_channels": nch,
                "num_fault_channels": 0,
                "accuracy": acc,
                "latency_first": latent_times['first_bacth'],
                "latency_second": latent_times['second_bacth'],
                "latency_avg": latent_times['mean'],
                "total_time": t_total
            })

            del(y_true, y_pred)
            gc.collect()   

    del(emg_data, label)
    gc.collect()
    
df_results = pd.DataFrame(results_list)
df_results.to_csv(f"{model_name}_experiment_1.csv", index=False)

1 out of 17 files
num windows 1667
-8.942352 7.3267903
Máximo valor da amplitude do ruído:  0.8
Número de canais com ruído:  0


2025-08-10 16:27:00.757811: I tensorflow/core/platform/cpu_feature_guard.cc:152] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE3 SSE4.1 SSE4.2 AVX
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


latencia primeiro batch =  1.4796137809753418
latencia segundo batch =  0.06773972511291504
tempo_total 129.14231085777283
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.9652069586082783
Número de canais com ruído:  1
latencia primeiro batch =  1.448822259902954
latencia segundo batch =  0.09273171424865723
tempo_total 257.2460403442383
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  2
latencia primeiro batch =  1.1588385105133057
latencia segundo batch =  0.1254739761352539
tempo_total 384.4418547153473
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  3
latencia primeiro batch =  1.1811721324920654
latencia segundo batch =  0.07365250587463379
tempo_total 512.4353351593018
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  4
latencia primeiro batch =  1.4839329719543457
latencia segundo batch =  0.0722188949584961
tempo_total 641.9218084812164
User: /root/data/data/65_HDsEMG/2

In [18]:
latent_times

{'first_bacth': 1.2789678573608398,
 'second_bacth': 0.07913064956665039,
 'mean': 0.07791341001857306}

In [19]:
cont = 1

sub = 0
acc_total = 0

params['plot_data'] = False
results_list=[]

for file in files_list[1:2]:
    print(f'{cont} out of {len(files_list)} files', end='\r')
    print()

    with h5py.File(file, 'r') as hf:
        emg_selected_extensor_data = hf['emg_extensors'][...].reshape(64, -1).astype(np.float32)
        emg_selected_flexor_data = hf['emg_flexors'][...].reshape(64, -1).astype(np.float32)
        label = hf['adjusted_class'][...]

    # print('data', emg_selected_extensor_data.shape, emg_selected_flexor_data.shape)
    if params['armband'] == True:
        
        idx = [np.arange(0,121,8), np.arange(1,58,8),np.arange(2,59,8), np.arange(5,62,8), np.arange(6,63,8), np.arange(7,129,8)]
        idx = np.hstack(idx)
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data), axis=0)
        emg_data = np.delete(emg_data,idx,axis=0)
        
    else:
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data),axis=0)

        
    
    idx_gestures = []
    
    for gest in params['gesture_list'][4:5]:
        idx_gestures += [np.argwhere(label[0,:] == gest)]
 
    emg_data = emg_data[:,np.vstack(idx_gestures)[:,0]]
    label = label[:,np.vstack(idx_gestures)[:,0]][0]
    
    all_gestures = list(params['gesture_list'])
    all_gestures = np.sort(all_gestures)


    GestureRemap = {g: new_g for new_g, g in enumerate(all_gestures)}
    yData = np.array([GestureRemap[g] for g in label])

    params['num_windows'] = (emg_data.shape[1]-params['step'])//params['step']
    print('num windows',params['num_windows'])#,emg_data.shape)

    print(np.min(emg_data), np.max(emg_data))
    t_init = time.time()
  
   
    model_name = "trainings/EMG_65HDsEMG_2D_CNN_128f_128ch_SCE_normminmax_bysample_adamW"    
    
    index1 = np.append(np.arange(8),np.arange(8))
    index1 = np.append(index1,index1)
    index1 = np.append(index1,index1)

    index2 = np.repeat(np.arange(8),4)
    index2 = np.append(index2,np.roll(index2,-4))
    
    for max_var in np.arange(8,13):
    
        print('Máximo valor da amplitude do ruído: ', max_var/10)

        for nch in np.arange(8,64,8):

            print('Número de canais com ruído: ',nch)
            
            (y_pred, y_true, latent_times) = predict_gesture(model_name, emg_data.copy(), yData, params, index1[:nch], index2[:nch], max_var/10, nch )

            t_total = time.time() - t_init
            print('tempo_total', t_total)

            # cont += 1
            acc = skm.accuracy_score(y_true, y_pred)
            print("User:",file,"Accuracy:", acc)

            # save the results
            results_list.append({
                "file": file,
                "gesture": gest,
                "max_noise_amplitude": max_var/10,
                "num_noisy_channels": nch,
                "num_fault_channels": 0,
                "accuracy": acc,
                "latency_first": latent_times['first_bacth'],
                "latency_second": latent_times['second_bacth'],
                "latency_avg": latent_times['mean'],
                "total_time": t_total
            })

            del(y_true, y_pred)
            gc.collect()   

    del(emg_data, label)
    gc.collect()
    
df_results = pd.DataFrame(results_list)
df_results.to_csv(f"{model_name}_experiment_2.csv", index=False)
 

1 out of 17 files
num windows 1667
-8.942352 7.3267903
Máximo valor da amplitude do ruído:  0.8
Número de canais com ruído:  8
latencia primeiro batch =  1.2745230197906494
latencia segundo batch =  0.06474542617797852
tempo_total 133.13451218605042
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  16
latencia primeiro batch =  1.5707015991210938
latencia segundo batch =  0.08252954483032227
tempo_total 264.71476697921753
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.004199160167966407
Número de canais com ruído:  24
latencia primeiro batch =  1.4237666130065918
latencia segundo batch =  0.0681309700012207
tempo_total 392.852655172348
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  32
latencia primeiro batch =  2.8719940185546875
latencia segundo batch =  0.10842227935791016
tempo_total 524.4931423664093
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  40
latencia primeiro ba

In [20]:
cont = 1

sub = 0
acc_total = 0

params['plot_data'] = False
results_list=[]

for file in files_list[1:2]:
    print(f'{cont} out of {len(files_list)} files', end='\r')
    print()

    with h5py.File(file, 'r') as hf:
        emg_selected_extensor_data = hf['emg_extensors'][...].reshape(64, -1).astype(np.float32)
        emg_selected_flexor_data = hf['emg_flexors'][...].reshape(64, -1).astype(np.float32)
        label = hf['adjusted_class'][...]

    # print('data', emg_selected_extensor_data.shape, emg_selected_flexor_data.shape)
    if params['armband'] == True:
        
        idx = [np.arange(0,121,8), np.arange(1,58,8),np.arange(2,59,8), np.arange(5,62,8), np.arange(6,63,8), np.arange(7,129,8)]
        idx = np.hstack(idx)
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data), axis=0)
        emg_data = np.delete(emg_data,idx,axis=0)
        
    else:
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data),axis=0)

        
    
    idx_gestures = []
    
    for gest in params['gesture_list'][4:5]:
        idx_gestures += [np.argwhere(label[0,:] == gest)]
 
    # print(np.vstack(idx_gestures).shape)
    emg_data = emg_data[:,np.vstack(idx_gestures)[:,0]]
    label = label[:,np.vstack(idx_gestures)[:,0]][0]
    
    all_gestures = list(params['gesture_list'])
    all_gestures = np.sort(all_gestures)


    GestureRemap = {g: new_g for new_g, g in enumerate(all_gestures)}
    yData = np.array([GestureRemap[g] for g in label])

    params['num_windows'] = (emg_data.shape[1]-params['step'])//params['step']
    print('num windows',params['num_windows'])#,emg_data.shape)

    print(np.min(emg_data), np.max(emg_data))
    t_init = time.time()
  
   
    model_name = "trainings/EMG_65HDsEMG_2D_CNN_128f_128ch_SCE_normminmax_bysample_adamW"
    
    index1 = np.append(np.arange(8),np.arange(8))
    index1 = np.append(index1,index1)
    index1 = np.append(index1,index1)

    index2 = np.repeat(np.arange(8),4)
    index2 = np.append(index2,np.roll(index2,-4))
    
    for max_var in np.arange(1):
    
        for nch in np.arange(8):

            print('Número de canais zerados: ',nch)
            
            (y_pred, y_true, latent_times) = predict_gesture(model_name, emg_data.copy(), yData, params, index1[:nch], index2[:nch], 0, nch )

            t_total = time.time() - t_init
            print('tempo_total', t_total)

            # cont += 1
            acc = skm.accuracy_score(y_true, y_pred)
            print("User:",file,"Accuracy:", acc)

            # save the results
            results_list.append({
                "file": file,
                "gesture": gest,
                "max_noise_amplitude": max_var/10,
                "num_noisy_channels": 0,
                "num_fault_channels": nch,
                "accuracy": acc,
                "latency_first": latent_times['first_bacth'],
                "latency_second": latent_times['second_bacth'],
                "latency_avg": latent_times['mean'],
                "total_time": t_total
            })

            del(y_true, y_pred)
            gc.collect()   

    del(emg_data, label)
    gc.collect()
    
df_results = pd.DataFrame(results_list)
df_results.to_csv(f"{model_name}_experiment_3.csv", index=False)

1 out of 17 files
num windows 1667
-8.942352 7.3267903
Número de canais zerados:  0
latencia primeiro batch =  1.655066967010498
latencia segundo batch =  0.08673214912414551
tempo_total 133.28573775291443
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.9652069586082783
Número de canais zerados:  1
latencia primeiro batch =  1.2341883182525635
latencia segundo batch =  0.10333991050720215
tempo_total 264.173864364624
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.9610077984403119
Número de canais zerados:  2
latencia primeiro batch =  1.8100545406341553
latencia segundo batch =  0.1702892780303955
tempo_total 395.3585443496704
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.9592081583683263
Número de canais zerados:  3
latencia primeiro batch =  1.3764903545379639
latencia segundo batch =  0.07239222526550293
tempo_total 528.0089154243469
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.9628074385122976
Número de canais zerados:  4
latencia primeiro batch =  1.22728

In [21]:
params['n_voting'] = 5

cont = 1

sub = 0
acc_total = 0

params['plot_data'] = False
results_list=[]

for file in files_list[1:2]:
    print(f'{cont} out of {len(files_list)} files', end='\r')
    print()

    with h5py.File(file, 'r') as hf:
        emg_selected_extensor_data = hf['emg_extensors'][...].reshape(64, -1).astype(np.float32)
        emg_selected_flexor_data = hf['emg_flexors'][...].reshape(64, -1).astype(np.float32)
        label = hf['adjusted_class'][...]

    # print('data', emg_selected_extensor_data.shape, emg_selected_flexor_data.shape)
    if params['armband'] == True:
        
        idx = [np.arange(0,121,8), np.arange(1,58,8),np.arange(2,59,8), np.arange(5,62,8), np.arange(6,63,8), np.arange(7,129,8)]
        idx = np.hstack(idx)
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data), axis=0)
        emg_data = np.delete(emg_data,idx,axis=0)
        
    else:
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data),axis=0)

        
    
    idx_gestures = []
    
    for gest in params['gesture_list'][4:5]:
        idx_gestures += [np.argwhere(label[0,:] == gest)]
 
    emg_data = emg_data[:,np.vstack(idx_gestures)[:,0]]
    label = label[:,np.vstack(idx_gestures)[:,0]][0]
    
    all_gestures = list(params['gesture_list'])
    all_gestures = np.sort(all_gestures)


    GestureRemap = {g: new_g for new_g, g in enumerate(all_gestures)}
    yData = np.array([GestureRemap[g] for g in label])

    params['num_windows'] = (emg_data.shape[1]-params['step'])//params['step']
    print('num windows',params['num_windows'])#,emg_data.shape)

    print(np.min(emg_data), np.max(emg_data))
    t_init = time.time()
  
   
    model_name = "trainings/EMG_65HDsEMG_2D_CNN_128f_128ch_SCE_normminmax_bysample_adamW"
    
    index1 = np.append(np.arange(8),np.arange(8))
    index1 = np.append(index1,index1)
    index1 = np.append(index1,index1)

    index2 = np.repeat(np.arange(8),4)
    index2 = np.append(index2,np.roll(index2,-4))
    
    for max_var in np.arange(8,13):
    
        print('Máximo valor da amplitude do ruído: ', max_var/10)

        for nch in np.arange(8):

            print('Número de canais com ruído: ',nch)
            
            (y_pred, y_true, latent_times) = predict_gesture(model_name, emg_data.copy(), yData, params, index1[:nch], index2[:nch], max_var/10, nch )

            t_total = time.time() - t_init
            print('tempo_total', t_total)

            # cont += 1
            acc = skm.accuracy_score(y_true, y_pred)
            print("User:",file,"Accuracy:", acc)

            # save the results
            results_list.append({
                "file": file,
                "gesture": gest,
                "max_noise_amplitude": max_var/10,
                "num_noisy_channels": nch,
                "num_fault_channels": 0,
                "accuracy": acc,
                "latency_first": latent_times['first_bacth'],
                "latency_second": latent_times['second_bacth'],
                "latency_avg": latent_times['mean'],
                "total_time": t_total
            })

            del(y_true, y_pred)
            gc.collect()   

    del(emg_data, label)
    gc.collect()
    
df_results = pd.DataFrame(results_list)
df_results.to_csv(f"{model_name}_experiment_4.csv", index=False)

1 out of 17 files
num windows 1667
-8.942352 7.3267903
Máximo valor da amplitude do ruído:  0.8
Número de canais com ruído:  0
latencia primeiro batch =  1.2068920135498047
latencia segundo batch =  0.06895565986633301
tempo_total 133.1769459247589
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.9652069586082783
Número de canais com ruído:  1
latencia primeiro batch =  2.393381118774414
latencia segundo batch =  0.11109423637390137
tempo_total 265.2555010318756
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  2
latencia primeiro batch =  1.2624175548553467
latencia segundo batch =  0.06900382041931152
tempo_total 401.1583716869354
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  3
latencia primeiro batch =  1.4446754455566406
latencia segundo batch =  0.06897997856140137
tempo_total 533.83869099617
User: /root/data/data/65_HDsEMG/25295225 Accuracy: 0.0
Número de canais com ruído:  4
latencia primeiro batch =  1.

In [22]:
params['n_voting'] = 5

cont = 1
params['plot_data'] = False
params['armband'] = True
results_list=[]

sub = 0
acc_total = 0

for file in files_list[1:2]:
    print(f'{cont} out of {len(files_list)} files', end='\r')
    print()

    with h5py.File(file, 'r') as hf:
        emg_selected_extensor_data = hf['emg_extensors'][...].reshape(64, -1).astype(np.float32)
        emg_selected_flexor_data = hf['emg_flexors'][...].reshape(64, -1).astype(np.float32)
        all_label = hf['adjusted_class'][...]

    # print('data', emg_selected_extensor_data.shape, emg_selected_flexor_data.shape)
    if params['armband'] == True:
        
        idx = [np.arange(0,121,8), np.arange(1,58,8),np.arange(2,59,8), np.arange(5,62,8), np.arange(6,63,8), np.arange(7,129,8)]
        idx = np.hstack(idx)
        all_emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data), axis=0)
        all_emg_data = np.delete(all_emg_data,idx,axis=0)
        
    else:
        emg_data = np.concatenate((emg_selected_extensor_data, emg_selected_flexor_data),axis=0)

        
    all_gestures = list(params['gesture_list'])
    all_gestures = np.sort(all_gestures)


    GestureRemap = {g: new_g for new_g, g in enumerate(all_gestures)}
    
    idx_gestures = []
    
    for gest in params['gesture_list']:
        idx_gestures += [np.argwhere(all_label[0,:] == gest)]

        # print(np.vstack(idx_gestures).shape)
        emg_data = all_emg_data[:,np.vstack(idx_gestures)[:,0]]
        label = all_label[:,np.vstack(idx_gestures)[:,0]][0]

        yData = np.array([GestureRemap[g] for g in label])

        params['num_windows'] = (emg_data.shape[1]-params['step'])//params['step']
        print('num windows',params['num_windows'])#,emg_data.shape)

        t_init = time.time()


        model_name = "trainings/EMG_65HDsEMG_2D_CNN_128f_128ch_SCE_normminmax_bysample_adamW"
        # model_name = model_path +'1SGD_0.01'

        (y_pred, y_true, latent_times) = predict_gesture(model_name, emg_data, yData, params, [],[],  1, 0)

        # print(y_pred,y_true)

        t_total = time.time() - t_init
        print('tempo_total', t_total)

        cont += 1
        acc = skm.accuracy_score(y_true, y_pred)
        print("User:",file,"Accuracy:", acc)

        acc_total += acc 
        sub += 1
        # del(y_true, y_pred, emg_data, label)
        # gc.collect() 
        # save the results
        results_list.append({
            "file": file,
            "gesture": gest,
            "max_noise_amplitude": max_var/10,
            "num_noisy_channels": 0,
            "num_fault_channels": 0,
            "accuracy": acc,
            "latency_first": latent_times['first_bacth'],
            "latency_second": latent_times['second_bacth'],
            "latency_avg": latent_times['mean'],
            "total_time": t_total
        })
print('acurácia total: ', acc_total/sub)
   
df_results = pd.DataFrame(results_list)
df_results.to_csv(f"{model_name}_experiment_5.csv", index=False)
   


1 out of 17 files
num windows 111798


ValueError: all the input array dimensions for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 128 and the array at index 1 has size 64

In [ ]:
print('acurácia total: ', acc_total/sub)
